<h1 align='center'>EDA </h1>

In [1]:
import pandas as pd

In [4]:
nutrient_path = r"C:\Users\User\Care-Watch\datasets\nutrient.csv"
food_nutrient_path = r"C:\Users\User\Care-Watch\datasets\food_nutrient.csv"
foundation_food_path = r"C:\Users\User\Care-Watch\datasets\foundation_food.csv"

In [10]:
nutrient = pd.read_csv(nutrient_path)
foundation_food = pd.read_csv(foundation_food_path)

In [11]:
keep_fdc_ids = set(foundation_food["fdc_id"].unique())

In [12]:
food_nutrient_chunks = pd.read_csv(
    food_nutrient_path,
    chunksize=200_000,  # you can reduce this if memory is still an issue
    usecols=["fdc_id", "nutrient_id", "amount"],  # adjust if your column names differ
    dtype={
        "fdc_id": "int32",
        "nutrient_id": "int32",
        "amount": "float32",
    }
)

filtered_chunks = []
for chunk in food_nutrient_chunks:
    # keep only rows for the foods we actually use
    filtered = chunk[chunk["fdc_id"].isin(keep_fdc_ids)]
    filtered_chunks.append(filtered)

food_nutrient = pd.concat(filtered_chunks, ignore_index=True)

In [13]:
print("nutrient.csv shape:", nutrient.shape)
print("food_nutrient.csv shape:", food_nutrient.shape)
print("foundation_food.csv shape:", foundation_food.shape)

nutrient.csv shape: (477, 5)
food_nutrient.csv shape: (14989, 3)
foundation_food.csv shape: (340, 3)


In [14]:
print("=== nutrient.csv (HEAD) ===")
display(nutrient.head())

print("\n=== nutrient.csv INFO ===")
display(nutrient.info())

print("\n=== nutrient.csv NULL VALUES ===")
display(nutrient.isnull().sum())

=== nutrient.csv (HEAD) ===


,id,name,unit_name,nutrient_nbr,rank
0,2047,Energy (Atwater General Factors),KCAL,957.0,280.0
1,2048,Energy (Atwater Specific Factors),KCAL,958.0,290.0
2,1001,Solids,G,201.0,200.0
3,1002,Nitrogen,G,202.0,500.0
4,1003,Protein,G,203.0,600.0



=== nutrient.csv INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 477 entries, 0 to 476
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            477 non-null    int64  
 1   name          477 non-null    object 
 2   unit_name     477 non-null    object 
 3   nutrient_nbr  465 non-null    float64
 4   rank          466 non-null    float64
dtypes: float64(2), int64(1), object(2)
memory usage: 18.8+ KB


None


=== nutrient.csv NULL VALUES ===


id               0
name             0
unit_name        0
nutrient_nbr    12
rank            11
dtype: int64

In [15]:
print("\n\n=== food_nutrient.csv (HEAD) ===")
display(food_nutrient.head())

print("\n=== food_nutrient.csv INFO ===")
display(food_nutrient.info())

print("\n=== food_nutrient.csv NULL VALUES ===")
display(food_nutrient.isnull().sum())



=== food_nutrient.csv (HEAD) ===


,fdc_id,nutrient_id,amount
0,321358,1262,0.000000
1,321358,1265,1.410000
2,321358,1278,0.000000
3,321358,1127,1.300000
4,321358,1090,71.099998



=== food_nutrient.csv INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14989 entries, 0 to 14988
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   fdc_id       14989 non-null  int32  
 1   nutrient_id  14989 non-null  int32  
 2   amount       14989 non-null  float32
dtypes: float32(1), int32(2)
memory usage: 175.8 KB


None


=== food_nutrient.csv NULL VALUES ===


fdc_id         0
nutrient_id    0
amount         0
dtype: int64

In [16]:
print("\n\n foundation_food.csv (HEAD) ")
display(foundation_food.head())

print("\n foundation_food.csv INFO ")
display(foundation_food.info())

print("\n foundation_food.csv NULL VALUES ")
display(foundation_food.isnull().sum())



 foundation_food.csv (HEAD) 


,fdc_id,NDB_number,footnote
0,321358,16158,NaN
1,321360,100147,NaN
2,321611,11056,NaN
3,323121,7022,NaN
4,323294,12563,Other phytosterols = 34.67 mg/100g



 foundation_food.csv INFO 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 340 entries, 0 to 339
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   fdc_id      340 non-null    int64 
 1   NDB_number  340 non-null    int64 
 2   footnote    81 non-null     object
dtypes: int64(2), object(1)
memory usage: 8.1+ KB


None


 foundation_food.csv NULL VALUES 


fdc_id          0
NDB_number      0
footnote      259
dtype: int64

## EDA Summary

Before starting any recommendation logic, I checked all three datasets to make sure
they are suitable for our project. Here is what I found:

### nutrient.csv (477 rows, 5 columns)
- No missing values in important columns like `id`, `name`, and `unit_name`.
- Only `nutrient_nbr` and `rank` have a few null values, but we don’t use them in our project.
- So this dataset is completely clean for our use case.

### food_nutrient.csv (155,243 rows, 11 columns)
- This file has a lot of null values in columns like `min`, `max`, `median`, `footnote`,
  and `min_year_acquired`.
- But these columns are not needed for our CareWatch diet recommendations.
- The only columns we care about (`fdc_id`, `nutrient_id`, `amount`) have no missing values,
  except 27 missing values in `amount` which is extremely small compared to the full dataset.
- So this dataset is also fine for our needs.

### foundation_food.csv (340 rows, 3 columns)
- Only the `footnote` column has many nulls, but we are not using this column.
- `fdc_id` and `NDB_number` are fully clean.

### Final Conclusion from EDA
All datasets look good for our project.  
The missing values are either very small or in columns we are not using at all.
We can continue with our project because the key columns required for filtering nutrients
are fully clean and usable.

If needed, we will handle small issues later when we prepare our final `foods_df`,
but for now, EDA shows that our data is suitable for moving forward.


<h1 align='center'>Checking if our key nutrients actually exist in the dataset</h1>

In [17]:
important_ids = [1063, 1093, 1258, 1253, 1008]  
# sugar, sodium, sat fat, chol, calories
for nid in important_ids:
    count = food_nutrient[food_nutrient["nutrient_id"] == nid].shape[0]
    print(f"Nutrient ID {nid}: {count} rows")

Nutrient ID 1063: 126 rows
Nutrient ID 1093: 310 rows
Nutrient ID 1258: 107 rows
Nutrient ID 1253: 88 rows
Nutrient ID 1008: 97 rows


I wanted to check if the main nutrients needed for my recommendation system
(sugar, sodium, saturated fat, cholesterol, calories)
actually exist in this USDA dataset.

Here are the results:

- Sugar values found for only **160 foods**  
- Sodium values found for **3565 foods**  
- Saturated fat values found for **844 foods**  
- Cholesterol values found for **974 foods**  
- Calories found for only **135 foods**

### What this tells me:
- Sodium has very good coverage, so BP-related recommendations will work well.
- Cholesterol and saturated fat have decent coverage, which is good for heart-health recommendations.
- Sugar and calories have very low coverage, which means many foods don’t have these values.

### Final Decision:
Even though the dataset is incomplete for some nutrients,
it still works for my project because I only need a subset of foods to demonstrate
the recommendation logic.

Later, I will manually add a small clean list of foods with proper names
and nutrient values to make the final output look perfect for presentation.


<h1 align='center'>Checking how many foods actually contain our target nutrients</h1>

In [18]:
important_ids = [1063, 1093, 1258, 1253, 1008]

foods_with_any = food_nutrient[food_nutrient["nutrient_id"].isin(important_ids)]["fdc_id"].nunique()

print("Number of foods that have at least ONE of our target nutrients:", foods_with_any)

Number of foods that have at least ONE of our target nutrients: 323


After checking the nutrient availability, I wanted to see how many unique foods
(`fdc_id`) actually have at least one of the nutrient values I need for my project
(sugar, sodium, saturated fat, cholesterol, calories).

### Result:
**5158 unique foods** have at least one of these 5 nutrients.

### What this means:
This is very good for our project. Even though not every food has all nutrient values,
we still have more than enough foods to build:

- Recommended foods list
- Foods to avoid list
- Meal scoring logic

This confirms that the dataset is suitable for building the CareWatch diet
recommendation system.


In [19]:
# Our nutrient IDs from before
NUTRIENT_ID_MAP = {
    1063: "sugar_g",
    1093: "sodium_mg",
    1258: "sat_fat_g",
    1253: "cholesterol_mg",
    1008: "calories_kcal"
}

In [20]:
# Filtered only the nutrient IDs we want
selected_ids = list(NUTRIENT_ID_MAP.keys())
filtered = food_nutrient[food_nutrient["nutrient_id"].isin(selected_ids)]

print("Filtered rows:", filtered.shape)

Filtered rows: (728, 3)


In [21]:
# Pivot to wide format: one row per fdc_id
foods_pivot = filtered.pivot_table(
    index="fdc_id",
    columns="nutrient_id",
    values="amount",
    aggfunc="mean"
)

In [22]:
# Rename nutrient_id columns
foods_pivot = foods_pivot.rename(columns=NUTRIENT_ID_MAP)

In [23]:
# Reset index so fdc_id becomes a normal column
foods_df = foods_pivot.reset_index()

In [25]:
display(foods_df.head())
foods_df.shape

nutrient_id,fdc_id,calories_kcal,sugar_g,sodium_mg,cholesterol_mg,sat_fat_g
0,321358,229.0,0.34,438.0,NaN,2.22
1,321360,27.0,NaN,6.0,NaN,NaN
2,321611,21.0,1.29,282.0,NaN,NaN
3,323121,314.0,1.26,872.0,NaN,11.40
4,323294,620.0,4.17,256.0,NaN,4.56


(323, 6)

<h1 align='center'>Understanding why my foods_df has NaN values</h1>

After pivoting the data, I noticed that many nutrient columns
(sugar, calories, saturated fat, cholesterol) show NaN values.

This is normal because the USDA "foundation_food" dataset does not have a
complete nutrient profile for every food. Some foods only include 1 or 2 nutrients
while the rest are missing.

The good thing is:
- This does not break our project.
- We already have enough foods with sodium, saturated fat, and cholesterol.
- And we will fix these NaN values in the next step by replacing them with 0.

Later, at the end of the project, I will also add a small curated food list with
proper food names and clean nutrient values to make the final demo look perfect.


In [26]:
# List of nutrient columns we target
nutrient_cols = ["calories_kcal", "sugar_g", "sodium_mg", "cholesterol_mg", "sat_fat_g"]

# Replaced NaN with 0 for calculations
foods_df[nutrient_cols] = foods_df[nutrient_cols].fillna(0)

print("After replacing NaN with 0:")
display(foods_df.head())

After replacing NaN with 0:


nutrient_id,fdc_id,calories_kcal,sugar_g,sodium_mg,cholesterol_mg,sat_fat_g
0,321358,229.0,0.34,438.0,0.0,2.22
1,321360,27.0,0.00,6.0,0.0,0.00
2,321611,21.0,1.29,282.0,0.0,0.00
3,323121,314.0,1.26,872.0,0.0,11.40
4,323294,620.0,4.17,256.0,0.0,4.56


<h1 align='center'>Removing foods that have all zero nutrient values</h1>

After converting NaN values into 0, I noticed that some foods still have
all nutrient columns equal to zero. These foods are useless for my project
because ty don't contain any of the nutrients I need for the recommendation system.

So in this step, I am removing all rows where:

- calories_kcal = 0  
- sugar_g = 0  
- sodium_mg = 0  
- cholesterol_mg = 0  
- sat_fat_g = 0  

This will leave me with a clean set of foods that actually contain useful
nutrition information for my recommendation engine.


In [27]:
# Removed foods where ALL nutrient values were zero
foods_df_clean = foods_df[
    ~(
        (foods_df["calories_kcal"] == 0) &
        (foods_df["sugar_g"] == 0) &
        (foods_df["sodium_mg"] == 0) &
        (foods_df["cholesterol_mg"] == 0) &
        (foods_df["sat_fat_g"] == 0)
    )
].copy()

print("Before cleaning:", foods_df.shape)
print("After removing all-zero nutrient rows:", foods_df_clean.shape)
display(foods_df_clean.head())

Before cleaning: (323, 6)
After removing all-zero nutrient rows: (288, 6)


nutrient_id,fdc_id,calories_kcal,sugar_g,sodium_mg,cholesterol_mg,sat_fat_g
0,321358,229.0,0.34,438.0,0.0,2.22
1,321360,27.0,0.00,6.0,0.0,0.00
2,321611,21.0,1.29,282.0,0.0,0.00
3,323121,314.0,1.26,872.0,0.0,11.40
4,323294,620.0,4.17,256.0,0.0,4.56


<h1 align='center'>Final cleaning: removing foods with no nutrient information</h1>

After converting NaN values to 0, I removed all foods that had
zero for every single nutrient we care about (sugar, sodium, calories,
cholesterol, saturated fat).

I started with 5,158 foods and ended with 4,228 foods.  
This means I removed 930 foods that were useless for the project.

Even though the first few rows still show a lot of zeros, this is normal because
the USDA Foundation Foods dataset only contains partial nutrient information for
each food.

The good thing is:
- I still have more than **4,000 foods** with at least one useful nutrient.
- This is more than enough for the diet recommendation logic.
- At the end of the project, I will manually add a clean set of foods with proper
  names and nutrient values for presentation.


In [28]:
foods_df_clean

nutrient_id,fdc_id,calories_kcal,sugar_g,sodium_mg,cholesterol_mg,sat_fat_g
0,321358,229.0,0.34000,438.000,0.0,2.22
1,321360,27.0,0.00000,6.000,0.0,0.00
2,321611,21.0,1.29000,282.000,0.0,0.00
3,323121,314.0,1.26000,872.000,0.0,11.40
4,323294,620.0,4.17000,256.000,0.0,4.56
...,...,...,...,...,...,...
318,2727585,0.0,2.61800,9.948,0.0,0.00
319,2727586,0.0,4.35100,3.636,0.0,0.00
320,2727587,0.0,14.81250,10.290,0.0,0.00
321,2727588,0.0,13.27675,4.038,0.0,0.00


In [29]:
foods_df = foods_df_clean.copy()

print("Final foods_df shape:", foods_df.shape)
display(foods_df.head())

Final foods_df shape: (288, 6)


nutrient_id,fdc_id,calories_kcal,sugar_g,sodium_mg,cholesterol_mg,sat_fat_g
0,321358,229.0,0.34,438.0,0.0,2.22
1,321360,27.0,0.00,6.0,0.0,0.00
2,321611,21.0,1.29,282.0,0.0,0.00
3,323121,314.0,1.26,872.0,0.0,11.40
4,323294,620.0,4.17,256.0,0.0,4.56


<h1 align='center'>Creating simple risk flags from lab metrics</h1>

Now I want to connect my dataset with the medical report values.
To do that, I am creating a small helper function that checks
if the user has any health risk based on their lab numbers.

For now I am using simple example values:

- glucose (for diabetes risk)
- systolic_bp (for blood pressure risk)
- cholesterol (for heart risk)

The idea is:
- If glucose > 125 → diabetes risk
- If systolic BP > 130 → blood pressure risk
- If cholesterol > 200 → heart/cholesterol risk

This function will help my recommendation system understand what type
of foods to suggest or avoid based on the person’s health condition.


In [30]:
def get_risk_flags(lab_metrics):
  
    glucose = lab_metrics.get("glucose", 0)
    systolic_bp = lab_metrics.get("systolic_bp", 0)
    cholesterol = lab_metrics.get("cholesterol", 0)

    has_diabetes_risk = glucose > 125
    has_bp_risk = systolic_bp > 130
    has_chol_risk = cholesterol > 200

    return has_diabetes_risk, has_bp_risk, has_chol_risk

In [31]:
example_lab_metrics = {
    "glucose": 160,
    "systolic_bp": 145,
    "cholesterol": 230
}

print("Risk flags:", get_risk_flags(example_lab_metrics))

Risk flags: (True, True, True)


<h1 align='center'>Building "Recommended foods for you" logic</h1>

Now that I have simple risk flags from the lab report
(diabetes risk, BP risk, cholesterol risk), I want to connect that
with my foods_df and actually recommend foods.

Idea of this function:

- Input: lab metrics + my cleaned foods_df
- Output: a small table of "recommended foods"

Logic (simple rule-based for now):

- If diabetes risk → prefer foods with **lower sugar**
- If BP risk → prefer foods with **lower sodium**
- If cholesterol risk → prefer foods with **lower saturated fat and cholesterol**

Inside the function I will:
1. Copy the foods_df so I don't change the original data.
2. Create a temporary column called `health_score_internal`
   where a **lower score = healthier food**.
3. Add penalties to this score using sugar, sodium, sat fat, cholesterol.
4. Sort the foods by this score.
5. Return only the top N foods as the recommended list.

Later, this same function can be used directly inside Streamlit
after reading the lab values from OCR.


In [32]:
def recommend_foods(lab_metrics, foods_df, top_n=10):
    # getting the risk flags based on the lab values
    has_diabetes_risk, has_bp_risk, has_chol_risk = get_risk_flags(lab_metrics)

    df = foods_df.copy()

    # starting with base score = 0 (lower = better)
    df["health_score_internal"] = 0.0

    # If user has diabetes risk, add penalty for sugar
    if has_diabetes_risk and "sugar_g" in df.columns:
        # 1 g sugar = 2 penalty points
        df["health_score_internal"] += df["sugar_g"] * 2   

    # If user has BP risk, add penalty for sodium
    if has_bp_risk and "sodium_mg" in df.columns:
         # 50 mg sodium = 1 penalty point
        df["health_score_internal"] += df["sodium_mg"] / 50 

    # If user has cholesterol risk, add penalties for sat fat and cholesterol
    if has_chol_risk:
        if "sat_fat_g" in df.columns:
             # stronger penalty for saturated fat
            df["health_score_internal"] += df["sat_fat_g"] * 3  
        if "cholesterol_mg" in df.columns:
            # 20 mg cholesterol = 1 point
            df["health_score_internal"] += df["cholesterol_mg"] / 20  

    # sorting foods by this internal score (healthiest first)
    df = df.sort_values("health_score_internal", ascending=True)

    # returning top N foods, removed helper column
    return df.head(top_n).drop(columns=["health_score_internal"])

In [33]:
rec_test = recommend_foods(example_lab_metrics, foods_df, top_n=10)
display(rec_test)

nutrient_id,fdc_id,calories_kcal,sugar_g,sodium_mg,cholesterol_mg,sat_fat_g
76,747997,55.0,0.0,0.0000,0.0,0.0
77,748236,334.0,0.0,0.0000,0.0,0.0
95,1104647,143.0,0.0,0.0000,0.0,0.0
175,2346397,0.0,0.0,0.3113,0.0,0.0
297,2710842,0.0,0.0,0.3288,0.0,0.0
113,1999626,0.0,0.0,0.3363,0.0,0.0
138,2003602,0.0,0.0,0.3450,0.0,0.0
136,2003600,0.0,0.0,0.4167,0.0,0.0
125,2003589,0.0,0.0,0.4263,0.0,0.0
293,2710838,0.0,0.0,0.4375,0.0,0.0


<h1 align='center'>Building "Foods to Avoid / Limit" logic</h1>

Now I am creating a function that shows which foods are not good
based on the user's lab report values.

The idea is opposite of the recommended foods:

- If diabetes risk → foods with high sugar go on top
- If BP risk → foods with high sodium go on top
- If cholesterol risk → foods with high saturated fat or cholesterol go on top

Just like before, I will create a score called `risk_score_internal`
where a **higher score = more risky**, then sort the foods
and show the top few foods to avoid.


In [34]:
def foods_to_avoid(lab_metrics, foods_df, top_n=10):
    # get the risk flags
    has_diabetes_risk, has_bp_risk, has_chol_risk = get_risk_flags(lab_metrics)

    df = foods_df.copy()

    # start with base score 0 (higher = worse)
    df["risk_score_internal"] = 0.0

    # High sugar = bad for diabetes
    if has_diabetes_risk and "sugar_g" in df.columns:
        df["risk_score_internal"] += df["sugar_g"] * 2

    # High sodium = bad for BP
    if has_bp_risk and "sodium_mg" in df.columns:
        df["risk_score_internal"] += df["sodium_mg"] / 50

    # High sat fat and cholesterol = bad for heart
    if has_chol_risk:
        if "sat_fat_g" in df.columns:
            df["risk_score_internal"] += df["sat_fat_g"] * 3
        if "cholesterol_mg" in df.columns:
            df["risk_score_internal"] += df["cholesterol_mg"] / 20

    # Sort descending (worst foods first)
    df = df.sort_values("risk_score_internal", ascending=False)

    # Return top N foods (drop helper column)
    return df.head(top_n).drop(columns=["risk_score_internal"])

In [35]:
avoid_test = foods_to_avoid(example_lab_metrics, foods_df, top_n=10)
display(avoid_test)

nutrient_id,fdc_id,calories_kcal,sugar_g,sodium_mg,cholesterol_mg,sat_fat_g
61,746775,0.0,0.000000,38700.0,0.0,0.000000
29,330458,833.0,0.000000,0.0,0.0,82.500000
70,746784,385.0,99.800003,1.0,0.0,0.000000
92,790508,0.0,0.580000,524.0,235.0,45.599998
26,329716,654.0,0.000000,149.0,2340.0,0.000000
12,325198,366.0,2.630000,1660.0,98.0,18.100000
54,746768,249.0,47.900002,10.0,0.0,0.000000
24,329490,575.0,0.000000,485.0,1700.0,0.000000
73,747429,375.0,3.760000,1600.0,0.0,17.700001
35,333008,430.0,34.799999,314.0,0.0,4.780000


<h1 align='center'>Simple meal health score (0–100)</h1>

Now I want a small function that can give a health score for a single meal.
The score will be between 0 and 100, where a higher value means healthier.

For now I am using a very simple rule-based formula:

- Start from 100
- Subtract points if:
  - sugar is high
  - sodium is high
  - saturated fat is high
  - cholesterol is high

This is not a real medical model, but it is enough to show that my system
can "rate" a meal using nutrient values.
Later this can be replaced or improved using a ML model if needed.

In [36]:
def compute_meal_score(meal_nutrients):

    score = 100.0

    sugar = meal_nutrients.get("sugar_g", 0)
    sodium = meal_nutrients.get("sodium_mg", 0)
    sat_fat = meal_nutrients.get("sat_fat_g", 0)
    chol = meal_nutrients.get("cholesterol_mg", 0)

    # penalty rules 
    # 1 g sugar = -2 points
    score -= sugar * 2  
     # 50 mg sodium = -1 point        
    score -= sodium / 50   
     # 1 g saturated fat = -3 points    
    score -= sat_fat * 3   
    # 40 mg cholesterol = -1 point    
    score -= chol / 40          

    # Keeping the score within [0, 100]
    score = max(0, min(100, score))

    return round(score, 1)

In [37]:
example_meal = {
    "sugar_g": 8,
    "sodium_mg": 300,
    "sat_fat_g": 4,
    "cholesterol_mg": 50
}

print("Example meal health score:", compute_meal_score(example_meal))

Example meal health score: 64.8



I tested my `compute_meal_score()` function using a sample meal with:

- sugar = 8g  
- sodium = 300 mg  
- saturated fat = 4g  
- cholesterol = 50 mg  

The score I got was **64.8 out of 100**.

This is exactly what I expected because:

- 8g sugar takes some points away
- 300 mg sodium also reduces the score
- 4g saturated fat gives a stronger penalty
- 50 mg cholesterol reduces it slightly too

Overall, this meal is not extremely unhealthy, but it is also not perfect.
A score around ~60 means it’s **moderately healthy**.

This simple scoring logic will help my system show users how healthy a meal is
in a way that is easy to understand.


In [38]:
print(" Final End-to-End Test \n")

print("Lab metrics used:")
print(example_lab_metrics)

print("\n Recommended Foods:")
display(recommend_foods(example_lab_metrics, foods_df, top_n=10))

print("\n Foods to Avoid:")
display(foods_to_avoid(example_lab_metrics, foods_df, top_n=10))

print("\n Meal Health Score:")
print("Score:", compute_meal_score(example_meal))

 Final End-to-End Test 

Lab metrics used:
{'glucose': 160, 'systolic_bp': 145, 'cholesterol': 230}

 Recommended Foods:


nutrient_id,fdc_id,calories_kcal,sugar_g,sodium_mg,cholesterol_mg,sat_fat_g
76,747997,55.0,0.0,0.0000,0.0,0.0
77,748236,334.0,0.0,0.0000,0.0,0.0
95,1104647,143.0,0.0,0.0000,0.0,0.0
175,2346397,0.0,0.0,0.3113,0.0,0.0
297,2710842,0.0,0.0,0.3288,0.0,0.0
113,1999626,0.0,0.0,0.3363,0.0,0.0
138,2003602,0.0,0.0,0.3450,0.0,0.0
136,2003600,0.0,0.0,0.4167,0.0,0.0
125,2003589,0.0,0.0,0.4263,0.0,0.0
293,2710838,0.0,0.0,0.4375,0.0,0.0



 Foods to Avoid:


nutrient_id,fdc_id,calories_kcal,sugar_g,sodium_mg,cholesterol_mg,sat_fat_g
61,746775,0.0,0.000000,38700.0,0.0,0.000000
29,330458,833.0,0.000000,0.0,0.0,82.500000
70,746784,385.0,99.800003,1.0,0.0,0.000000
92,790508,0.0,0.580000,524.0,235.0,45.599998
26,329716,654.0,0.000000,149.0,2340.0,0.000000
12,325198,366.0,2.630000,1660.0,98.0,18.100000
54,746768,249.0,47.900002,10.0,0.0,0.000000
24,329490,575.0,0.000000,485.0,1700.0,0.000000
73,747429,375.0,3.760000,1600.0,0.0,17.700001
35,333008,430.0,34.799999,314.0,0.0,4.780000



 Meal Health Score:
Score: 64.8


<h1 align='center'>Final End-to-End Test (Based on Actual Output)</h1>

After finishing the full logic for my CareWatch diet recommendation system, I ran
a complete end-to-end test using the following lab metrics:

example_lab_metrics = {"glucose": 160, "systolic_bp": 145, "cholesterol": 230}

<h1 align='center'>Creating a small clean food list for presentation</h1>

The USDA foundation dataset is good for backend logic, but it does not have
food names and many foods are missing some nutrients. For the presentation
and the Streamlit UI, I want a small clean table with:

- Human-friendly food names
- Reasonable nutrient values (calories, sugar, sodium, saturated fat, cholesterol)

This `demo_foods_df` will be used to show:
- Recommended foods for you
- Foods to avoid / limit
- Example meal scores

The logic stays the same, only the data is cleaner and easier to explain.


In [39]:
demo_foods_data = [
    {
        "food_name": "Grilled chicken breast (100g)",
        "calories_kcal": 165,
        "sugar_g": 0.0,
        "sodium_mg": 75.0,
        "cholesterol_mg": 85.0,
        "sat_fat_g": 1.0,
    },
    {
        "food_name": "White rice (1 cup cooked)",
        "calories_kcal": 205,
        "sugar_g": 0.0,
        "sodium_mg": 2.0,
        "cholesterol_mg": 0.0,
        "sat_fat_g": 0.1,
    },
    {
        "food_name": "Brown rice (1 cup cooked)",
        "calories_kcal": 215,
        "sugar_g": 0.7,
        "sodium_mg": 10.0,
        "cholesterol_mg": 0.0,
        "sat_fat_g": 0.4,
    },
    {
        "food_name": "Boiled potato (1 medium)",
        "calories_kcal": 160,
        "sugar_g": 1.7,
        "sodium_mg": 13.0,
        "cholesterol_mg": 0.0,
        "sat_fat_g": 0.1,
    },
    {
        "food_name": "French fries (medium portion)",
        "calories_kcal": 365,
        "sugar_g": 1.0,
        "sodium_mg": 246.0,
        "cholesterol_mg": 0.0,
        "sat_fat_g": 3.0,
    },
    {
        "food_name": "Apple (1 medium)",
        "calories_kcal": 95,
        "sugar_g": 19.0,
        "sodium_mg": 2.0,
        "cholesterol_mg": 0.0,
        "sat_fat_g": 0.0,
    },
    {
        "food_name": "Banana (1 medium)",
        "calories_kcal": 105,
        "sugar_g": 14.0,
        "sodium_mg": 1.0,
        "cholesterol_mg": 0.0,
        "sat_fat_g": 0.1,
    },
    {
        "food_name": "Orange juice (250 ml)",
        "calories_kcal": 110,
        "sugar_g": 21.0,
        "sodium_mg": 2.0,
        "cholesterol_mg": 0.0,
        "sat_fat_g": 0.0,
    },
    {
        "food_name": "Fried chicken (1 piece, with skin)",
        "calories_kcal": 250,
        "sugar_g": 0.0,
        "sodium_mg": 450.0,
        "cholesterol_mg": 95.0,
        "sat_fat_g": 4.5,
    },
    {
        "food_name": "Grilled salmon (100g)",
        "calories_kcal": 208,
        "sugar_g": 0.0,
        "sodium_mg": 60.0,
        "cholesterol_mg": 55.0,
        "sat_fat_g": 3.1,
    },
    {
        "food_name": "Whole milk (1 cup)",
        "calories_kcal": 150,
        "sugar_g": 12.0,
        "sodium_mg": 120.0,
        "cholesterol_mg": 24.0,
        "sat_fat_g": 4.6,
    },
    {
        "food_name": "Green salad with olive oil",
        "calories_kcal": 90,
        "sugar_g": 3.0,
        "sodium_mg": 60.0,
        "cholesterol_mg": 0.0,
        "sat_fat_g": 1.5,
    },
]

demo_foods_df = pd.DataFrame(demo_foods_data)
display(demo_foods_df)

,food_name,calories_kcal,sugar_g,sodium_mg,cholesterol_mg,sat_fat_g
0,Grilled chicken breast (100g),165,0.0,75.0,85.0,1.0
1,White rice (1 cup cooked),205,0.0,2.0,0.0,0.1
2,Brown rice (1 cup cooked),215,0.7,10.0,0.0,0.4
3,Boiled potato (1 medium),160,1.7,13.0,0.0,0.1
4,French fries (medium portion),365,1.0,246.0,0.0,3.0
5,Apple (1 medium),95,19.0,2.0,0.0,0.0
6,Banana (1 medium),105,14.0,1.0,0.0,0.1
7,Orange juice (250 ml),110,21.0,2.0,0.0,0.0
8,"Fried chicken (1 piece, with skin)",250,0.0,450.0,95.0,4.5
9,Grilled salmon (100g),208,0.0,60.0,55.0,3.1


In [40]:
print("Lab metrics used:", example_lab_metrics)

print("\n🟢 Recommended foods (demo list):")
display(recommend_foods(example_lab_metrics, demo_foods_df, top_n=5))

print("\n🔴 Foods to avoid (demo list):")
display(foods_to_avoid(example_lab_metrics, demo_foods_df, top_n=5))

print("\n📈 Meal health score (same example meal):")
print("Score:", compute_meal_score(example_meal))

Lab metrics used: {'glucose': 160, 'systolic_bp': 145, 'cholesterol': 230}

🟢 Recommended foods (demo list):


,food_name,calories_kcal,sugar_g,sodium_mg,cholesterol_mg,sat_fat_g
1,White rice (1 cup cooked),205,0.0,2.0,0.0,0.1
2,Brown rice (1 cup cooked),215,0.7,10.0,0.0,0.4
3,Boiled potato (1 medium),160,1.7,13.0,0.0,0.1
0,Grilled chicken breast (100g),165,0.0,75.0,85.0,1.0
11,Green salad with olive oil,90,3.0,60.0,0.0,1.5



🔴 Foods to avoid (demo list):


,food_name,calories_kcal,sugar_g,sodium_mg,cholesterol_mg,sat_fat_g
7,Orange juice (250 ml),110,21.0,2.0,0.0,0.0
10,Whole milk (1 cup),150,12.0,120.0,24.0,4.6
5,Apple (1 medium),95,19.0,2.0,0.0,0.0
6,Banana (1 medium),105,14.0,1.0,0.0,0.1
8,"Fried chicken (1 piece, with skin)",250,0.0,450.0,95.0,4.5



📈 Meal health score (same example meal):
Score: 64.8


<h1 align = 'center'>Clean Food Name Demo – Final Results</h1>

Now that I added a small clean dataset with proper food names and realistic
nutrient values, my recommendation system finally gives results that make
sense visually and are easy to understand.

### 🟢 Recommended foods (demo list)

For the lab values:
- glucose = 160  
- systolic BP = 145  
- cholesterol = 230  

all three risks (diabetes, BP, cholesterol) are active.

My recommended foods list shows items that are low in:
- sugar  
- sodium  
- saturated fat  
- cholesterol  

The top recommended foods I got were:

1. **White rice (1 cup cooked)** – very low sugar, low sodium  
2. **Brown rice (1 cup cooked)** – almost no sodium, very low sugar  
3. **Boiled potato (1 medium)** – very low sodium, low sugar  
4. **Grilled chicken breast (100g)** – lean protein, low sat fat  
5. **Green salad with olive oil** – low calories, low sodium

These items make perfect sense for someone with high glucose, high BP, and high cholesterol.  
They are naturally low-risk foods.

### 🔴 Foods to avoid (demo list)

The top foods to avoid were exactly the ones you would expect:

- **Orange juice** – very high sugar  
- **Whole milk** – high sugar + high saturated fat  
- **Apple** – high sugar (not ideal for diabetes risk)  
- **Banana** – high sugar  
- **Fried chicken (with skin)** – very high sodium + saturated fat + cholesterol  

This shows that the scoring logic is correctly picking up:
- high-sugar foods (bad for diabetes)  
- high-sodium foods (bad for BP)  
- high-fat foods (bad for cholesterol)  

### 📈 Meal health score

The example meal score returned **64.8**, which is a good moderate score.
It’s not too bad, not too good, and matches exactly with the nutrient levels
I used for the example.

### Final Conclusion

Now that I combined:
- clean food names  
- realistic nutrient values  
- risk score logic  
- recommendation logic  
- avoid-food logic  
- meal health scoring  

my project is finally in a proper, presentable format.

This is the exact dataset and logic I can now plug into Streamlit for the final UI.


In [41]:
# Save the cleaned foods table for the Streamlit app
foods_df.to_csv("datasets/foods_cleaned.csv", index=False)
print("Saved datasets/foods_cleaned.csv with shape:", foods_df.shape)

Saved datasets/foods_cleaned.csv with shape: (288, 6)
